In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-8"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
schema1 = 'loglevel string , logtime string'

In [3]:
logdf = spark.read.format("csv").schema(schema1).load("/public/trendytech/datasets/logdata1m.csv")

In [4]:
logdf1 = logdf.withColumn("logtime",to_timestamp("logtime"))  ## convert string to timestamp

In [5]:
logdf1.createOrReplaceTempView("serverlogs")

## SQL Style code for pivot, window, rank etc

#### pivot

In [20]:
##The PIVOT clause needs a subquery first (Spark can't pivot directly off a raw table reference) — that's the inner SELECT producing loglevel, month.

In [6]:
spark.sql("""
  SELECT * FROM (
    SELECT loglevel, date_format(logtime, 'MMMM') AS month
    FROM serverlogs
  )
  PIVOT (
    COUNT(*) 
    FOR month IN ('January','February','March','April','May','June',
                   'July','August','September','October','November','December')
  )
""").show()

+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+
|loglevel|January|February|March|April|  May| June| July|August|September|October|November|December|
+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+
|    INFO|  29119|   28983|29095|29302|28900|29143|29300| 28993|    29038|  29018|   23301|   28874|
|   ERROR|   4054|    4013| 4122| 4107| 4086| 4059| 3976|  3987|     4161|   4040|    3389|    4106|
|    WARN|   8217|    8266| 8165| 8277| 8403| 8191| 8222|  8381|     8352|   8226|    6616|    8328|
|   FATAL|     94|      72|   70|   83|   60|   78|   98|    80|       81|     92|   16797|      94|
|   DEBUG|  41961|   41734|41652|41869|41785|41774|42085| 42147|    41433|  41936|   33366|   41749|
+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+



#### running total

In [8]:
windowDF = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/public/trendytech/datasets/windowdatamodified.csv")

In [10]:
windowDF.createOrReplaceTempView("windowDF")

In [11]:
spark.sql("""
  SELECT country, weeknum, invoicevalue,
         SUM(invoicevalue) OVER (
           PARTITION BY country 
           ORDER BY weeknum 
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
         ) AS running_tot
  FROM windowDF
""").show()

+-------+-------+------------+------------------+
|country|weeknum|invoicevalue|       running_tot|
+-------+-------+------------+------------------+
| Sweden|     50|      2646.3|            2646.3|
|Germany|     48|      1600.0|            1600.0|
|Germany|     49|      1800.0|            3400.0|
|Germany|     50|      1800.0|            5200.0|
|Germany|     51|      1600.0|            6800.0|
| France|     48|       500.0|             500.0|
| France|     49|       500.0|            1000.0|
| France|     50|      537.32|1537.3200000000002|
| France|     51|       500.0|2037.3200000000002|
|Belgium|     48|       800.0|             800.0|
|Belgium|     50|      625.16|1425.1599999999999|
|Belgium|     51|       800.0|           2225.16|
|Finland|     50|       892.8|             892.8|
|  India|     48|       300.0|             300.0|
|  India|     49|      3284.1|            3584.1|
|  India|     50|     2321.78|           5905.88|
|  India|     51|       300.0|           6205.88|


### rank, lag 

In [12]:
spark.sql("""
  SELECT country, weeknum, invoicevalue,
         RANK() OVER (PARTITION BY country ORDER BY invoicevalue DESC) AS rank_in_country,
         LAG(invoicevalue, 1) OVER (PARTITION BY country ORDER BY weeknum) AS prev_week_value
  FROM windowDF
""").show()

+-------+-------+------------+---------------+---------------+
|country|weeknum|invoicevalue|rank_in_country|prev_week_value|
+-------+-------+------------+---------------+---------------+
| Sweden|     50|      2646.3|              1|           null|
|Germany|     48|      1600.0|              3|           null|
|Germany|     49|      1800.0|              1|         1600.0|
|Germany|     50|      1800.0|              1|         1800.0|
|Germany|     51|      1600.0|              3|         1800.0|
| France|     48|       500.0|              2|           null|
| France|     49|       500.0|              2|          500.0|
| France|     50|      537.32|              1|          500.0|
| France|     51|       500.0|              2|         537.32|
|Belgium|     48|       800.0|              1|           null|
|Belgium|     50|      625.16|              3|          800.0|
|Belgium|     51|       800.0|              1|         625.16|
|Finland|     50|       892.8|              1|         

In [21]:
## pivot and rank  further examples

In [17]:
spark.sql("""
  SELECT * FROM (
    SELECT loglevel, date_format(logtime, 'MMMM') AS month
    FROM serverlogs
  )
  PIVOT (
    COUNT(*)
    FOR month IN ('January','February','March','April','May','June',
                   'July','August','September','October','November','December')
  )
""").show()

+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+
|loglevel|January|February|March|April|  May| June| July|August|September|October|November|December|
+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+
|    INFO|  29119|   28983|29095|29302|28900|29143|29300| 28993|    29038|  29018|   23301|   28874|
|   ERROR|   4054|    4013| 4122| 4107| 4086| 4059| 3976|  3987|     4161|   4040|    3389|    4106|
|    WARN|   8217|    8266| 8165| 8277| 8403| 8191| 8222|  8381|     8352|   8226|    6616|    8328|
|   DEBUG|  41961|   41734|41652|41869|41785|41774|42085| 42147|    41433|  41936|   33366|   41749|
|   FATAL|     94|      72|   70|   83|   60|   78|   98|    80|       81|     92|   16797|      94|
+--------+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+



In [18]:
spark.sql("""
  SELECT loglevel, month, log_count,
         RANK() OVER (PARTITION BY loglevel ORDER BY log_count DESC) AS busiest_month_rank
  FROM (
    SELECT loglevel, date_format(logtime, 'MMMM') AS month, COUNT(*) AS log_count
    FROM serverlogs
    GROUP BY loglevel, date_format(logtime, 'MMMM')
  )
  ORDER BY loglevel, busiest_month_rank
""").show()

+--------+---------+---------+------------------+
|loglevel|    month|log_count|busiest_month_rank|
+--------+---------+---------+------------------+
|   DEBUG|   August|    42147|                 1|
|   DEBUG|     July|    42085|                 2|
|   DEBUG|  January|    41961|                 3|
|   DEBUG|  October|    41936|                 4|
|   DEBUG|    April|    41869|                 5|
|   DEBUG|      May|    41785|                 6|
|   DEBUG|     June|    41774|                 7|
|   DEBUG| December|    41749|                 8|
|   DEBUG| February|    41734|                 9|
|   DEBUG|    March|    41652|                10|
|   DEBUG|September|    41433|                11|
|   DEBUG| November|    33366|                12|
|   ERROR|September|     4161|                 1|
|   ERROR|    March|     4122|                 2|
|   ERROR|    April|     4107|                 3|
|   ERROR| December|     4106|                 4|
|   ERROR|      May|     4086|                 5|


### combining rank & pivot

In [19]:
spark.sql("""
  WITH monthly AS (
    SELECT loglevel, date_format(logtime, 'MMMM') AS month, COUNT(*) AS log_count
    FROM serverlogs
    GROUP BY loglevel, date_format(logtime, 'MMMM')
  ),
  ranked AS (
    SELECT loglevel, month, log_count,
           RANK() OVER (PARTITION BY loglevel ORDER BY log_count DESC) AS rnk
    FROM monthly
  )
  SELECT * FROM ranked
  PIVOT (
    FIRST(log_count)
    FOR month IN ('January','February','March','April','May','June',
                   'July','August','September','October','November','December')
  )
""").show()

+--------+---+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+
|loglevel|rnk|January|February|March|April|  May| June| July|August|September|October|November|December|
+--------+---+-------+--------+-----+-----+-----+-----+-----+------+---------+-------+--------+--------+
|    INFO|  1|   null|    null| null|29302| null| null| null|  null|     null|   null|    null|    null|
|    INFO|  2|   null|    null| null| null| null| null|29300|  null|     null|   null|    null|    null|
|    INFO|  3|   null|    null| null| null| null|29143| null|  null|     null|   null|    null|    null|
|    INFO|  4|  29119|    null| null| null| null| null| null|  null|     null|   null|    null|    null|
|    INFO|  5|   null|    null|29095| null| null| null| null|  null|     null|   null|    null|    null|
|    INFO|  6|   null|    null| null| null| null| null| null|  null|    29038|   null|    null|    null|
|    INFO|  7|   null|    null| null| null| null| null|